In [0]:
CREATE OR REPLACE VIEW dbx_joshdevph_dev.processed.vw_yellow_tripdata_dq
AS

WITH dq_checks AS (

    SELECT
        *,

        -- ============================================================
        -- DQ001: Completeness
        -- Pickup datetime must not be NULL
        -- ============================================================
        CASE
            WHEN tpep_pickup_datetime IS NOT NULL
            THEN TRUE
            ELSE FALSE
        END AS dq001_pickup_datetime_not_null,


        -- ============================================================
        -- DQ002: Completeness
        -- Dropoff datetime must not be NULL
        -- ============================================================
        CASE
            WHEN tpep_dropoff_datetime IS NOT NULL
            THEN TRUE
            ELSE FALSE
        END AS dq002_dropoff_datetime_not_null,


        -- ============================================================
        -- DQ003: Consistency
        -- Dropoff datetime must not be earlier than pickup datetime
        -- ============================================================
        CASE
            WHEN tpep_pickup_datetime IS NOT NULL
             AND tpep_dropoff_datetime IS NOT NULL
             AND tpep_dropoff_datetime >= tpep_pickup_datetime
            THEN TRUE
            ELSE FALSE
        END AS dq003_valid_trip_datetime,


        -- ============================================================
        -- DQ004: Reasonability
        -- Trip distance must not be negative
        -- ============================================================
        CASE
            WHEN trip_distance IS NOT NULL
             AND trip_distance >= 0
            THEN TRUE
            ELSE FALSE
        END AS dq004_trip_distance_non_negative,


        -- ============================================================
        -- DQ005: Reasonability
        -- Passenger count must be between 0 and 8
        -- ============================================================
        CASE
            WHEN passenger_count IS NOT NULL
             AND passenger_count BETWEEN 0 AND 8
            THEN TRUE
            ELSE FALSE
        END AS dq005_passenger_count_valid,


        -- ============================================================
        -- DQ006: Reasonability
        -- Fare amount must not be negative
        -- ============================================================
        CASE
            WHEN fare_amount IS NOT NULL
             AND fare_amount >= 0
            THEN TRUE
            ELSE FALSE
        END AS dq006_fare_amount_non_negative,


        -- ============================================================
        -- DQ007: Validity
        -- Payment type must contain a recognized code
        -- ============================================================
        CASE
            WHEN payment_type IS NOT NULL
             AND payment_type IN (0, 1, 2, 3, 4, 5, 6)
            THEN TRUE
            ELSE FALSE
        END AS dq007_payment_type_valid,


        -- ============================================================
        -- DQ008: Consistency
        -- Total amount should reconcile with its components
        -- Allow a tolerance of $0.01
        -- ============================================================
        CASE
            WHEN total_amount IS NOT NULL
             AND ABS(
                    total_amount -
                    (
                        COALESCE(fare_amount, 0)
                        + COALESCE(extra, 0)
                        + COALESCE(mta_tax, 0)
                        + COALESCE(tip_amount, 0)
                        + COALESCE(tolls_amount, 0)
                        + COALESCE(improvement_surcharge, 0)
                        + COALESCE(congestion_surcharge, 0)
                        + COALESCE(Airport_fee, 0)
                        + COALESCE(cbd_congestion_fee, 0)
                    )
                 ) <= 0.01
            THEN TRUE
            ELSE FALSE
        END AS dq008_total_amount_reconciles

    FROM dbx_joshdevph_dev.raw.stg_yellow_tripdata
),

dq_results AS (

    SELECT
        *,

        -- ============================================================
        -- Overall Data Quality Status
        -- ============================================================
        CASE
            WHEN dq001_pickup_datetime_not_null
             AND dq002_dropoff_datetime_not_null
             AND dq003_valid_trip_datetime
             AND dq004_trip_distance_non_negative
             AND dq005_passenger_count_valid
             AND dq006_fare_amount_non_negative
             AND dq007_payment_type_valid
             AND dq008_total_amount_reconciles
            THEN 'PASS'
            ELSE 'FAIL'
        END AS dq_status,


        -- ============================================================
        -- Number of failed DQ rules
        -- ============================================================
        (
            CASE WHEN NOT dq001_pickup_datetime_not_null THEN 1 ELSE 0 END +
            CASE WHEN NOT dq002_dropoff_datetime_not_null THEN 1 ELSE 0 END +
            CASE WHEN NOT dq003_valid_trip_datetime THEN 1 ELSE 0 END +
            CASE WHEN NOT dq004_trip_distance_non_negative THEN 1 ELSE 0 END +
            CASE WHEN NOT dq005_passenger_count_valid THEN 1 ELSE 0 END +
            CASE WHEN NOT dq006_fare_amount_non_negative THEN 1 ELSE 0 END +
            CASE WHEN NOT dq007_payment_type_valid THEN 1 ELSE 0 END +
            CASE WHEN NOT dq008_total_amount_reconciles THEN 1 ELSE 0 END
        ) AS dq_failed_rule_count,


        -- ============================================================
        -- List of failed DQ rules
        -- ============================================================
        ARRAY_COMPACT(
            ARRAY(
                CASE
                    WHEN NOT dq001_pickup_datetime_not_null
                    THEN 'DQ001'
                END,

                CASE
                    WHEN NOT dq002_dropoff_datetime_not_null
                    THEN 'DQ002'
                END,

                CASE
                    WHEN NOT dq003_valid_trip_datetime
                    THEN 'DQ003'
                END,

                CASE
                    WHEN NOT dq004_trip_distance_non_negative
                    THEN 'DQ004'
                END,

                CASE
                    WHEN NOT dq005_passenger_count_valid
                    THEN 'DQ005'
                END,

                CASE
                    WHEN NOT dq006_fare_amount_non_negative
                    THEN 'DQ006'
                END,

                CASE
                    WHEN NOT dq007_payment_type_valid
                    THEN 'DQ007'
                END,

                CASE
                    WHEN NOT dq008_total_amount_reconciles
                    THEN 'DQ008'
                END
            )
        ) AS dq_failed_rules

    FROM dq_checks
)

SELECT *
FROM dq_results
;